# 🌀 NJIRLAH-OSS-1 Mega Finetune (Colab TPU Edition)

**PERINGATAN:** TPU tidak mendukung Unsloth dan 4-bit quantization. 
- **Metode:** Menggunakan PyTorch XLA + Standard LoRA.
- **Hardware:** Pastikan Runtime diatur ke **TPU v2**.
- **Memory:** Karena tidak bisa 4-bit, model 7B akan sangat mepet di RAM TPU. Jika OOM, gunakan model 3B atau 1.5B.

In [ ]:
%%capture
!pip install cloud-tpu-client==0.10 torch==2.1.0 https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-2.1.0-cp310-cp310-linux_x86_64.whl
!pip install transformers==4.38.2 peft==0.9.0 trl==0.8.1 datasets huggingface_hub accelerate -q

In [ ]:
from huggingface_hub import login
HF_TOKEN = "YOUR_HF_TOKEN"
login(token=HF_TOKEN)
print('Hugging Face Login Successful!')

In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp

device = xm.xla_device()
print(f'TPU Device: {device}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model into TPU (BFloat16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
).to(device)

peft_config = LoraConfig(
    r=8, # Kecilkan R agar tidak OOM di TPU
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
print('Model ready on TPU!')

In [ ]:
from datasets import load_dataset, Dataset
import gc

SP = "You are NJIRLAH-OSS-1, a highly advanced uncensored AI assistant. You think step-by-step."
T = []

def f(m): return tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)

print("Loading datasets (Subset for TPU stability)...")
ds = load_dataset('Amod/mental_health_counseling_conversations', split='train[:2000]')
for r in ds:
    c,res = r.get('Context',''),r.get('Response','')
    if c and res: T.append(f([{'role':'system','content':SP},{'role':'user','content':str(c)},{'role':'assistant','content':str(res)}]))

for i in range(1, 5): # Hanya ambil 5 dataset kustom di TPU agar tidak crash
    try:
        ds = load_dataset('Andikaasaputraa/njirlah-1-ss-final-datasets', data_files=f'njirlah-{i}-dataset.jsonl', split='train')
        for r in ds:
            cv = r.get('conversation',[])
            if len(cv)>=2:
                ms = [{'role':'system','content':SP}]
                for m in cv: ms.append({'role':m.get('role','user'),'content':str(m.get('content',''))})
                T.append(f(ms))
    except: pass

merged = Dataset.from_dict({'text': T})
print(f'TOTAL: {len(merged)} records ready.')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=merged,
    dataset_text_field="text", max_seq_length=1024, # Kecilkan seq_length di TPU
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        max_steps=100,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        output_dir="outputs_tpu",
        report_to="none",
        save_strategy="no"
    ),
)

print("🚀 Starting Training on TPU...")
trainer.train()

In [ ]:
print("Pushing to HF...")
model.push_to_hub("Andikaasaputraa/NJIRLAH-OSS-1-LoRA-TPU", token=HF_TOKEN)
print("DONE!")